# Lab Trino – Silver to gold

In [1]:
import os
import pandas as pd
import trino

os.environ["TRINO_HOST"] = "trino"
os.environ["TRINO_PORT"] = "8080"
os.environ["TRINO_USER"] = os.environ.get("KUBERNETES_NAMESPACE", "user-k-sison-ece")

CATALOG = "lakehouse"

conn = trino.dbapi.connect(
    host=os.environ["TRINO_HOST"],
    port=int(os.environ["TRINO_PORT"]),
    user=os.environ["TRINO_USER"],
    http_scheme="http",
    catalog=CATALOG,
)

def sql(statement):
    cur = conn.cursor()
    cur.execute(statement)
    rows = cur.fetchall()  # the query only completes when its results are fetched
    columns = [c[0] for c in cur.description] if cur.description else []
    return pd.DataFrame(rows, columns=columns)

print(os.environ["TRINO_USER"])
sql("SELECT version()")

user-k-sison-ece


,_col0
0,480


## Environment 

In [2]:
sql("SHOW SCHEMAS")

,Schema
0,information_schema
1,silver
2,system


## Catalogs, schemas and tables

In [3]:
display(sql("SHOW CATALOGS"))
display(sql("SHOW TABLES FROM lakehouse.silver"))

,Catalog
0,lakehouse
1,system
2,tpcds
3,tpch


,Table
0,orders


In [4]:
display(sql("SHOW TABLES FROM lakehouse.silver"))
sql("SELECT count(*) AS users FROM lakehouse.silver.users")

,Table
0,orders
1,users


,users
0,57


In [5]:
display(sql("DESCRIBE lakehouse.silver.orders"))
display(sql("DESCRIBE lakehouse.silver.users"))

,Column,Type,Extra,Comment
0,order_id,varchar,,
1,user_id,varchar,,
2,ordered_at,timestamp(6) with time zone,,
3,quantity,integer,,
4,product,varchar,,
5,source,varchar,,


,Column,Type,Extra,Comment
0,user_id,varchar,,
1,username,varchar,,
2,name,varchar,,
3,sex,varchar,,
4,email,varchar,,
5,birthdate,date,,
6,street,varchar,,
7,city,varchar,,
8,state,varchar,,
9,zip_code,varchar,,


In [6]:
sql("SELECT ordered_at FROM lakehouse.silver.orders LIMIT 3")

,ordered_at
0,2020-01-28 12:31:23.938466+00:00
1,2020-04-18 22:51:09.653035+00:00
2,2020-02-24 01:03:36.630111+00:00


## Query the silver layer

In [7]:
display(sql("SELECT count(*) AS orders FROM lakehouse.silver.orders"))
display(sql("SELECT count(*) AS users FROM lakehouse.silver.users"))

,orders
0,2833


,users
0,57


In [8]:
sql("""
    SELECT state,
           count(*) AS users,
           count_if(city IS NULL) AS missing_city
    FROM lakehouse.silver.users
    GROUP BY state
    ORDER BY users DESC
    LIMIT 10
""")

,state,users,missing_city
0,AP,4,4
1,TX,3,0
2,KY,3,0
3,FM,3,0
4,AE,3,3
5,NJ,2,0
6,VI,2,0
7,PA,2,0
8,VT,2,0
9,NM,2,0


In [9]:
sql("""
    SELECT count(*) AS orphan_orders
    FROM lakehouse.silver.orders AS o
    LEFT JOIN lakehouse.silver.users AS u ON u.user_id = o.user_id
    WHERE u.user_id IS NULL
""")

,orphan_orders
0,0


## Iceberg metadata

In [10]:
sql("""
    SELECT snapshot_id, committed_at, operation,
           summary['engine-name'] AS engine,
           summary['added-records'] AS added_records
    FROM lakehouse.silver."orders$snapshots"
    ORDER BY committed_at
""")

,snapshot_id,committed_at,operation,engine,added_records
0,4219974134046780544,2026-10-28 15:48:20.462+00:00,overwrite,spark,2829
1,7719747662070971571,2026-11-03 19:54:57.000+00:00,append,spark,3
2,1220184473976417285,2026-11-03 19:59:47.577+00:00,append,spark,1


In [11]:
sql("""
    SELECT file_path, file_format, record_count, file_size_in_bytes
    FROM lakehouse.silver."orders$files"
""")

,file_path,file_format,record_count,file_size_in_bytes
0,s3a://user-k-sison-ece/warehouse/silver/orders...,PARQUET,1,2201
1,s3a://user-k-sison-ece/warehouse/silver/orders...,PARQUET,2829,84055
2,s3a://user-k-sison-ece/warehouse/silver/orders...,PARQUET,3,1956


In [12]:
sql('SELECT * FROM lakehouse.silver."orders$history"')

,made_current_at,snapshot_id,parent_id,is_current_ancestor
0,2026-10-28 15:48:20.462+00:00,4219974134046780544,NaN,True
1,2026-11-03 19:54:57.000+00:00,7719747662070971571,4.219974e+18,True
2,2026-11-03 19:59:47.577+00:00,1220184473976417285,7.719748e+18,True


## Query plan and execution

In [14]:
plan = sql("""
    EXPLAIN (TYPE DISTRIBUTED)
    SELECT u.state, count(*) AS orders
    FROM lakehouse.silver.orders AS o
    JOIN lakehouse.silver.users  AS u ON u.user_id = o.user_id
    GROUP BY u.state
""")
print(plan.iloc[0, 0])

Trino version: 480
Fragment 0 [HASH]
    Output layout: [state, count]
    Output partitioning: SINGLE []
    Output[columnNames = [state, orders]]
    │   Layout: [state:varchar, count:bigint]
    │   Estimates: {rows: ? (?), cpu: 0, memory: 0B, network: 0B}
    │   orders := count
    └─ Aggregate[type = FINAL, keys = [state]]
       │   Layout: [state:varchar, count:bigint]
       │   Estimates: {rows: ? (?), cpu: ?, memory: ?, network: 0B}
       │   count := count(count_4)
       └─ LocalExchange[partitioning = HASH, arguments = [state::varchar]]
          │   Layout: [state:varchar, count_4:bigint]
          │   Estimates: {rows: ? (?), cpu: ?, memory: 0B, network: 0B}
          └─ RemoteSource[sourceFragmentIds = [1]]
                 Layout: [state:varchar, count_4:bigint]

Fragment 1 [SOURCE]
    Output layout: [state, count_4]
    Output partitioning: HASH [state]
    Aggregate[type = PARTIAL, keys = [state]]
    │   Layout: [state:varchar, count_4:bigint]
    │   count_4 := 

In [16]:
try:
    sql("""
        EXPLAIN ANALYZE
        SELECT u.state, count(*) AS orders
        FROM lakehouse.silver.orders AS o
        JOIN lakehouse.silver.users  AS u ON u.user_id = o.user_id
        GROUP BY u.state
    """)
except Exception as e:
    print(getattr(e, "message", str(e)))

Failed to load table: orders in silver namespace


In [17]:
sql("""
    SELECT u.state, count(*) AS orders
    FROM lakehouse.silver.orders AS o
    JOIN lakehouse.silver.users  AS u ON u.user_id = o.user_id
    GROUP BY u.state
""")

TrinoExternalError: TrinoExternalError(type=EXTERNAL, name=ICEBERG_CATALOG_ERROR, message="Failed to load table: orders in silver namespace", query_id=20261124_212047_00021_ph3jd)